In [0]:
# Challenge 4 Setup: Performance Detective
# Creates tables with intentional performance characteristics
# for students to investigate and fix.

import re
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "main"
schema = f"challenge_4_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Schema: {catalog}.{schema}")

In [0]:
# --- Create order_history: a large table with MANY small files ---
# This simulates a table that receives frequent batch appends
# without any clustering or optimization.

random.seed(99)

regions = ['North', 'South', 'East', 'West']
channels = ['online', 'store', 'wholesale']
statuses = ['completed', 'shipped', 'returned', 'pending']

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("region", StringType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount_pct", DoubleType(), False),
    StructField("channel", StringType(), False),
    StructField("status", StringType(), False)
])

spark.sql("DROP TABLE IF EXISTS order_history")

# Initial load of 2000 rows
rows = []
for i in range(2000):
    order_date = date(2023, 1, 1) + timedelta(days=random.randint(0, 729))  # 2 years
    rows.append((
        i + 1,
        order_date,
        random.randint(1, 20),
        random.choice(regions),
        random.randint(1, 50),
        random.randint(1, 10),
        round(random.uniform(15, 300), 2),
        round(random.choice([0, 0, 0, 0.05, 0.10, 0.15, 0.20]), 2),
        random.choice(channels),
        random.choice(statuses)
    ))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("order_history")

# Add 10 more small batch appends to create file fragmentation
for batch in range(10):
    batch_rows = []
    for i in range(150):
        order_date = date(2023, 1, 1) + timedelta(days=random.randint(0, 729))
        batch_rows.append((
            2001 + batch * 150 + i,
            order_date,
            random.randint(1, 20),
            random.choice(regions),
            random.randint(1, 50),
            random.randint(1, 10),
            round(random.uniform(15, 300), 2),
            round(random.choice([0, 0, 0, 0.05, 0.10, 0.15, 0.20]), 2),
            random.choice(channels),
            random.choice(statuses)
        ))
    batch_df = spark.createDataFrame(batch_rows, schema=schema_def)
    batch_df.write.mode("append").saveAsTable("order_history")

total_orders = spark.sql("SELECT COUNT(*) FROM order_history").collect()[0][0]
print(f"Created order_history: {total_orders} rows (fragmented across many files)")

In [0]:
# --- Create customer_dim: small dimension table ---
spark.sql("DROP TABLE IF EXISTS customer_dim")

spark.sql("""
  CREATE TABLE customer_dim (
    customer_id INT,
    customer_name STRING,
    region STRING,
    segment STRING
  )
""")

segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
names = ['Acme Corp', 'Beta LLC', 'Gamma Inc', 'Delta Co', 'Epsilon Ltd',
         'Zeta Group', 'Eta Partners', 'Theta Intl', 'Iota Tech', 'Kappa Sys',
         'Lambda AI', 'Mu Digital', 'Nu Cloud', 'Xi Data', 'Omicron Net',
         'Pi Logic', 'Rho Analytics', 'Sigma BI', 'Tau Soft', 'Upsilon IO']

insert_values = []
for i, name in enumerate(names, 1):
    region = regions[(i - 1) % 4]
    segment = segments[(i - 1) % 4]
    insert_values.append(f"({i}, '{name}', '{region}', '{segment}')")

spark.sql(f"INSERT INTO customer_dim VALUES {', '.join(insert_values)}")
print("Created customer_dim: 20 rows")

In [0]:
# --- Create product_dim: medium dimension table ---
spark.sql("DROP TABLE IF EXISTS product_dim")

spark.sql("""
  CREATE TABLE product_dim (
    product_id INT,
    product_name STRING,
    category STRING,
    subcategory STRING,
    cost_price DOUBLE
  )
""")

categories = {
    'Electronics': ['Laptops', 'Phones', 'Tablets', 'Monitors', 'Accessories'],
    'Clothing': ['Mens', 'Womens', 'Kids', 'Shoes', 'Outerwear'],
    'Home': ['Furniture', 'Kitchen', 'Decor', 'Lighting', 'Storage'],
    'Food': ['Snacks', 'Beverages', 'Frozen', 'Organic', 'Bakery'],
    'Sports': ['Running', 'Cycling', 'Swimming', 'Gym', 'Outdoor']
}

product_id = 1
insert_values = []
for cat, subcats in categories.items():
    for subcat in subcats:
        for j in range(2):  # 2 products per subcategory = 50 total
            name = f"{subcat} Item {j+1}"
            cost = round(random.uniform(5, 150), 2)
            insert_values.append(f"({product_id}, '{name}', '{cat}', '{subcat}', {cost})")
            product_id += 1

spark.sql(f"INSERT INTO product_dim VALUES {', '.join(insert_values)}")
print("Created product_dim: 50 rows")

In [0]:
# --- Summary ---
print("="*60)
print("CHALLENGE 4 SETUP COMPLETE")
print("="*60)
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Tables created:")
print(f"  order_history  - {total_orders} rows, NO clustering, many small files")
print(f"  customer_dim   - 20 rows (dimension)")
print(f"  product_dim    - 50 rows (dimension)")
print(f"")
print(f"Next: Open the SQL Editor and run:")
print(f"  USE CATALOG {catalog};")
print(f"  USE SCHEMA {schema};")
print(f"")
print(f"Then work through the Challenge 4 notebook.")